# Notebook 5 — Live serving testbed with ground truth
**Question.** Do real client tools — a key-rotating OpenAI-SDK script, LiteLLM's Router, an agent fleet — leave the turn-taking signature when they hit a real LLM server, measured the way a provider would measure it (arrival → end of stream, at its own gateway)? And do honest users, batch jobs and an open-loop relay stay at R ≈ 1?

**Setup.** An open-weights model served by vLLM on the T4 (or, if vLLM cannot start, a simulated server whose latency rises with load). A logging gateway in front of it records every request by API key. One driver process runs all populations at once, each on its own keys:

| label | population | expected |
|---|---|---|
| `hum` | 400 simulated humans (sessions, think time) | R ≈ 1 |
| `bat` | 4 legitimate single-key batch jobs | R ≈ 1 when pooled with humans |
| `A1sem` | key-rotating script, 8 workers over 40 keys (OpenAI SDK) | R ≪ 1 |
| `A2jit` | same with up to 6 s random delay per request | R < 1, weaker |
| `A3llm` | LiteLLM Router, simple-shuffle over 40 keys, 8 workers | R ≪ 1 |
| `A4agt` | 6 agents, one key per episode, tool calls between steps | R < 1 |
| `A5rel` | open-loop relay, Poisson arrivals over 40 keys | R ≈ 1 (undetectable by design) |
| `Ccomp` | legitimate company, one queue over 10 keys | R ≪ 1 (designed false positive) |

Time is compressed: short generations (median 96 tokens) and short think times, so ~40 minutes give hundreds of request durations per key. The high-pass block is scaled to τ = 20 s.

**Pre-registered go/no-go (proposal, month 3).** Stop the occupancy track if the default harnesses (A1, A3) do not give R_τ < 0.9.

**Run settings.** Accelerator **GPU T4 ×1 (or ×2)**, Internet **On**. Runtime ≈ 60–75 min (≈ 10–15 of them installing vLLM).
Paste back the block between the PASTE markers.

In [ ]:
# Cell 1 — settings
import os, sys, json, time, subprocess, signal
T_START = time.time()
CFG = dict(
    backend="vllm",                    # "vllm" = stop if vLLM cannot start; "auto" = fall back to the simulated server; "sim"
    model="Qwen/Qwen2.5-0.5B-Instruct",
    run_s=int(os.environ.get("NB5_RUN_S", 2400)), warmup_s=int(os.environ.get("NB5_WARMUP", 180)), tau=20.0, seed=5,
    tok_med=96, tok_sig=0.6,
    human=dict(gap_s=float(os.environ.get("NB5_HUMAN_GAP", 360)), mean_req=6, think_med_s=6.0, think_sig=0.9),
    agent=dict(mean_steps=8, tool_med_s=2.0, tok_med=64),
    workloads=dict(humans=int(os.environ.get("NB5_HUMANS", 400)), batch_clients=4, A1_K=8, A2_K=8, A2_jitter_s=6.0,
                   A3_K=8, A4_agents=6, A5_rate=2.0, C_K=4),
    gateway="http://127.0.0.1:9000",
)
print(json.dumps(CFG, indent=1))


In [ ]:
%%writefile occlib.py
"""
occlib — occupancy statistics for detecting hidden closed-loop worker pools behind many accounts.

Data model: a request log as three equal-length arrays
    acct : int   account index 0..A-1
    s    : float request arrival time (seconds)
    e    : float time the streamed response finished (seconds), e >= s
X_i(t) = number of account i's requests in flight; N(t) = sum_i X_i(t).

All statistics are exact time integrals of step functions (no time grid).
"""
import numpy as np

__version__ = "0.1.0"

# --------------------------------------------------------------------------------------------
# core: exact per-group integrals of X and X^2
# --------------------------------------------------------------------------------------------
def _group_integrals(gid, s, e, ngroups):
    """For intervals carrying a group id, return I1[g] = int X_g dt and I2[g] = int X_g^2 dt,
    where X_g(t) = number of the group's intervals covering t (half-open [s, e))."""
    I1 = np.zeros(ngroups); I2 = np.zeros(ngroups)
    n = len(s)
    if n == 0:
        return I1, I2
    t = np.concatenate([s, e])
    d = np.concatenate([np.ones(n), -np.ones(n)])
    g = np.concatenate([gid, gid])
    o = np.lexsort((d, t, g))          # by group, then time, then ends before starts
    t, d, g = t[o], d[o], g[o]
    x = np.cumsum(d)                    # each group nets to zero, so the level resets between groups
    dt = np.empty_like(t)
    dt[:-1] = t[1:] - t[:-1]
    dt[-1] = 0.0
    dt[:-1][g[1:] != g[:-1]] = 0.0
    I1 += np.bincount(g, weights=x * dt, minlength=ngroups)
    I2 += np.bincount(g, weights=x * x * dt, minlength=ngroups)
    return I1, I2


def _clip(acct, s, e, t0, t1):
    s = np.clip(np.asarray(s, float), t0, t1); e = np.clip(np.asarray(e, float), t0, t1)
    m = e > s
    return np.asarray(acct)[m], s[m], e[m]


def _split_blocks(acct, s, e, t0, tau, nb):
    """Split intervals at block boundaries t0 + k*tau. Returns (acct, block, s, e) pieces."""
    b0 = np.floor((s - t0) / tau).astype(np.int64)
    b1 = np.floor((np.nextafter(e, -np.inf) - t0) / tau).astype(np.int64)
    b0 = np.clip(b0, 0, nb - 1); b1 = np.clip(b1, 0, nb - 1)
    cnt = b1 - b0 + 1
    idx = np.repeat(np.arange(len(s)), cnt)
    off = np.arange(cnt.sum()) - np.repeat(np.cumsum(cnt) - cnt, cnt)
    blk = b0[idx] + off
    ps = np.maximum(s[idx], t0 + blk * tau)
    pe = np.minimum(e[idx], t0 + (blk + 1) * tau)
    m = pe > ps
    return acct[idx][m], blk[m], ps[m], pe[m]


def occupancy(acct, s, e, t0, t1, tau=None, n_accounts=None):
    """Occupancy moments over [t0, t1].
    tau=None -> full-window variances; tau=float -> mean over blocks of within-block variances
    (the 'high-pass' statistic). Returns dict with R, V_N, V_i (per account), mean_N, n_blocks."""
    acct = np.asarray(acct, np.int64)
    A = int(n_accounts if n_accounts is not None else (acct.max() + 1 if len(acct) else 0))
    if tau is None:
        tau = t1 - t0
    nb = int(np.floor((t1 - t0) / tau + 1e-9))
    if nb < 1:
        raise ValueError("window shorter than one block")
    t1 = t0 + nb * tau
    a, s, e = _clip(acct, s, e, t0, t1)
    a, b, s, e = _split_blocks(a, s, e, t0, tau, nb)
    # aggregate N, per block
    I1, I2 = _group_integrals(b, s, e, nb)
    VN_b = I2 / tau - (I1 / tau) ** 2
    # per account per block (compress group ids)
    key = a * nb + b
    uk, inv = np.unique(key, return_inverse=True)
    J1, J2 = _group_integrals(inv, s, e, len(uk))
    Vg = J2 / tau - (J1 / tau) ** 2
    V_i = np.bincount((uk // nb).astype(np.int64), weights=Vg, minlength=A) / nb
    V_N = VN_b.mean()
    sV = V_i.sum()
    return dict(R=V_N / sV if sV > 0 else np.nan, V_N=V_N, V_i=V_i, sumV=sV,
                mean_N=I1.sum() / (nb * tau), n_blocks=nb, tau=tau)


def R_full(acct, s, e, t0, t1):
    return occupancy(acct, s, e, t0, t1, None)["R"]


def R_hp(acct, s, e, t0, t1, tau=60.0):
    return occupancy(acct, s, e, t0, t1, tau)["R"]


def fano_khat(acct, s, e, t0, t1):
    """Fano factor of N and the implied effective number of hidden slots K = <N>^2/(<N>-Var N)."""
    o = occupancy(acct, s, e, t0, t1, None)
    F = o["V_N"] / o["mean_N"] if o["mean_N"] > 0 else np.nan
    p = 1 - F
    K = o["mean_N"] / p if p > 0 else np.inf
    return dict(F=F, p_hat=p, K_hat=K, mean_N=o["mean_N"], V_N=o["V_N"])


def baselines(acct, s, e, t0, t1):
    """Simple baselines: requests/hour, peak concurrency, Fano factor of N."""
    a, sc, ec = _clip(acct, s, e, t0, t1)
    n_req = int(((np.asarray(s) >= t0) & (np.asarray(s) < t1)).sum())
    t = np.concatenate([sc, ec]); d = np.concatenate([np.ones(len(sc)), -np.ones(len(ec))])
    o = np.lexsort((d, t)); peak = int(np.cumsum(d[o]).max()) if len(t) else 0
    f = fano_khat(acct, s, e, t0, t1)
    return dict(req_per_h=n_req / ((t1 - t0) / 3600), peak_N=peak, fano=f["F"])


# --------------------------------------------------------------------------------------------
# nulls
# --------------------------------------------------------------------------------------------
def circular_shift(acct, s, e, t0, t1, rng):
    """Shift each account's timeline by an independent uniform offset, wrapping around [t0, t1).
    Keeps each account's marginal behaviour and autocorrelation; destroys cross-account dependence."""
    acct = np.asarray(acct, np.int64); L = t1 - t0
    a, s, e = _clip(acct, s, e, t0, t1)
    A = int(acct.max() + 1) if len(acct) else 0
    off = rng.uniform(0, L, A)[a]
    s2 = t0 + np.mod(s - t0 + off, L); e2 = s2 + (e - s)
    wrap = e2 > t1
    a_out = np.concatenate([a, a[wrap]])
    s_out = np.concatenate([s2, np.full(wrap.sum(), t0)])
    e_out = np.concatenate([np.minimum(e2, t1), t0 + (e2[wrap] - t1)])
    return a_out, s_out, e_out


def surrogate_null(acct, s, e, t0, t1, tau, n_sur, rng):
    """Distribution of R (tau=None) or R_tau under independent circular shifts."""
    return np.array([occupancy(*circular_shift(acct, s, e, t0, t1, rng), t0, t1, tau)["R"] for _ in range(n_sur)])


# --------------------------------------------------------------------------------------------
# pool growth: correlation of each candidate's occupancy with a pool's aggregate occupancy
# --------------------------------------------------------------------------------------------
def _cum_integral(s, e, t0, t1):
    """Knots and running integral F(t) = int_t0^t N(u) du for the superposition of intervals."""
    t = np.concatenate([[t0], s, e, [t1]])
    d = np.concatenate([[0.0], np.ones(len(s)), -np.ones(len(e)), [0.0]])
    o = np.lexsort((d, t)); t, d = t[o], d[o]
    x = np.cumsum(d)
    F = np.concatenate([[0.0], np.cumsum(x[:-1] * np.diff(t))])
    return t, F


def corr_with_pool(pool, cand, t0, t1, tau=None, n_cand=None):
    """pool=(acct,s,e) of pool members; cand=(acct,s,e) of candidates (acct = candidate index 0..C-1).
    Returns per-candidate correlation of X_j with N_pool (within-block if tau is set). Linear in traffic."""
    pa, ps, pe = _clip(*pool, t0, t1)
    ca, cs, ce = _clip(*cand, t0, t1)
    C = int(n_cand if n_cand is not None else np.asarray(cand[0]).max() + 1)
    if tau is None:
        tau = t1 - t0
    nb = int(np.floor((t1 - t0) / tau + 1e-9)); t1 = t0 + nb * tau
    # pool per-block mean and within-block variance
    _, pb, pps, ppe = _split_blocks(pa, np.clip(ps, t0, t1), np.clip(pe, t0, t1), t0, tau, nb)
    I1, I2 = _group_integrals(pb, pps, ppe, nb)
    Nbar = I1 / tau; VN = (I2 / tau - Nbar ** 2).mean()
    kn, F = _cum_integral(np.clip(ps, t0, t1), np.clip(pe, t0, t1), t0, t1)
    # candidate pieces
    ca2, cb, cs2, ce2 = _split_blocks(ca, np.clip(cs, t0, t1), np.clip(ce, t0, t1), t0, tau, nb)
    XN = np.interp(ce2, kn, F) - np.interp(cs2, kn, F)       # int over piece of N_pool
    key = ca2 * nb + cb
    uk, inv = np.unique(key, return_inverse=True)
    A = np.bincount(inv, weights=XN)                          # int X_j N over block
    B = np.bincount(inv, weights=ce2 - cs2)                   # int X_j over block
    jj = uk // nb; bb = uk % nb
    cov = np.bincount(jj, weights=A / tau - (B / tau) * Nbar[bb], minlength=C) / nb
    Vj = occupancy(ca, cs, ce, t0, t1, None if nb == 1 else tau, n_accounts=C)["V_i"]
    with np.errstate(invalid="ignore", divide="ignore"):
        return cov / np.sqrt(Vj * VN)


# --------------------------------------------------------------------------------------------
# generators (simulation)
# --------------------------------------------------------------------------------------------
DUR_MED, DUR_SIG = 8.0, 0.6       # streamed response duration, log-normal
GAP_MED, GAP_SIG = 0.4, 0.5       # client processing gap, log-normal
E_DUR = DUR_MED * np.exp(DUR_SIG ** 2 / 2)
E_GAP = GAP_MED * np.exp(GAP_SIG ** 2 / 2)


def duty(J):
    return E_DUR / (E_DUR + E_GAP + J / 2.0)


def theory_R(p, M):
    return (1 - p) / (1 - p / M)


def _pack(rows):
    if not rows:
        return np.zeros(0, np.int64), np.zeros(0), np.zeros(0)
    a, s, e = map(np.asarray, zip(*rows))
    return a.astype(np.int64), s.astype(float), e.astype(float)


def benign_humans(rng, n_accounts, T, session_rate=1 / 7200, mean_req=6, think_med=25.0, think_sig=0.9,
                  diurnal=0.0, offset=0):
    """Human-like accounts: Poisson session starts, geometric #requests, log-normal think time.
    diurnal in [0,1): amplitude of a 24 h sinusoidal rate modulation (common to all accounts)."""
    rows = []
    for i in range(n_accounts):
        t = rng.exponential(1 / session_rate)
        while t < T:
            if diurnal > 0 and rng.random() > (1 + diurnal * np.sin(2 * np.pi * t / 86400)) / (1 + diurnal):
                t += rng.exponential(1 / session_rate); continue
            tt = t
            for _ in range(rng.geometric(1 / mean_req)):
                if tt > T: break
                d = rng.lognormal(np.log(DUR_MED), DUR_SIG); rows.append((offset + i, tt, tt + d))
                tt += d + rng.lognormal(np.log(think_med), think_sig)
            t = tt + rng.exponential(1 / session_rate)
    return _pack(rows)


def closed_loop_pool(rng, T, K, M, J=0.0, assign="random", stint=None, K_bar=None, epoch=None, offset=0):
    """K closed-loop slots over M accounts. Each slot: send, wait for stream end, gap, uniform(0,J) delay.
    assign: 'random' (per request), 'pinned' (slot w -> account w % M), 'stint' (hold an account for `stint` s).
    K_bar/epoch: if set, the number of active slots is redrawn ~ Poisson(K_bar) (capped at K) every `epoch` s."""
    rows = []
    Kact = None
    if K_bar is not None:
        Kact = np.minimum(rng.poisson(K_bar, int(T // epoch) + 2), K)
    held = {}
    for w in range(K):
        t = rng.uniform(0, 5)
        while t < T:
            if Kact is not None:
                ei = int(t // epoch)
                if w >= Kact[ei]:
                    t = (ei + 1) * epoch + rng.uniform(0, 1); continue
            if assign == "pinned":
                a = w % M
            elif assign == "stint":
                k = (w, int(t // stint))
                if k not in held: held[k] = rng.integers(M)
                a = held[k]
            else:
                a = rng.integers(M)
            d = rng.lognormal(np.log(DUR_MED), DUR_SIG)
            rows.append((offset + a, t, t + d))
            t += d + rng.lognormal(np.log(GAP_MED), GAP_SIG) + (rng.uniform(0, J) if J > 0 else 0.0)
    return _pack(rows)


def open_loop_relay(rng, T, M, lam, policy="random", offset=0):
    """Open-loop Poisson arrivals (rate lam/s) dispatched over M accounts: random | rr | least."""
    import heapq
    rows = []; inflight = np.zeros(M, int); heap = []; t = 0.0; rr = 0
    while True:
        t += rng.exponential(1 / lam)
        if t > T: break
        while heap and heap[0][0] <= t:
            _, a = heapq.heappop(heap); inflight[a] -= 1
        if policy == "random": a = int(rng.integers(M))
        elif policy == "rr": a = rr % M; rr += 1
        else:
            m = inflight.min(); a = int(rng.choice(np.flatnonzero(inflight == m)))
        d = rng.lognormal(np.log(DUR_MED), DUR_SIG)
        rows.append((offset + a, t, t + d)); inflight[a] += 1; heapq.heappush(heap, (t + d, a))
    return _pack(rows)


def concat(*logs):
    return tuple(np.concatenate(x) for x in zip(*logs))


def relabel(acct):
    """Map arbitrary account ids to 0..A-1."""
    u, inv = np.unique(acct, return_inverse=True)
    return inv.astype(np.int64), u


In [ ]:
import importlib, occlib
importlib.reload(occlib)
print("occlib", occlib.__version__)


In [ ]:
%%writefile simserver.py
"""Simulated OpenAI-compatible streaming server (fallback when no GPU/vLLM).
Latency model: requests beyond `capacity` wait in a FIFO queue; per-token time grows with the number of active
sequences (batching interference): tpot = base_tpot * (1 + slowdown * active / capacity)."""
import asyncio, json, time, argparse, random
from aiohttp import web

ap = argparse.ArgumentParser()
ap.add_argument("--port", type=int, default=8000)
ap.add_argument("--capacity", type=int, default=128)
ap.add_argument("--base_tpot", type=float, default=0.02)
ap.add_argument("--slowdown", type=float, default=1.0)
ap.add_argument("--ttft", type=float, default=0.08)
args = ap.parse_args()
sem = asyncio.Semaphore(args.capacity)
active = 0

async def chat(request):
    global active
    body = await request.json()
    n = int(body.get("max_tokens") or 64)
    resp = web.StreamResponse(status=200, headers={"Content-Type": "text/event-stream"})
    await resp.prepare(request)
    async with sem:
        active += 1
        try:
            await asyncio.sleep(args.ttft * random.uniform(0.8, 1.2))
            for i in range(n):
                chunk = {"id": "x", "object": "chat.completion.chunk", "created": int(time.time()), "model": body.get("model", "sim"),
                         "choices": [{"index": 0, "delta": {"content": f"w{random.randint(0, 9999)} "}, "finish_reason": None}]}
                await resp.write(f"data: {json.dumps(chunk)}\n\n".encode())
                await asyncio.sleep(args.base_tpot * (1 + args.slowdown * active / args.capacity))
            fin = {"id": "x", "object": "chat.completion.chunk", "created": int(time.time()), "model": body.get("model", "sim"),
                   "choices": [{"index": 0, "delta": {}, "finish_reason": "length"}]}
            await resp.write(f"data: {json.dumps(fin)}\n\n".encode())
            await resp.write(b"data: [DONE]\n\n")
        finally:
            active -= 1
    await resp.write_eof()
    return resp

async def models(request):
    return web.json_response({"object": "list", "data": [{"id": "sim", "object": "model"}]})

async def health(request):
    return web.Response(text="ok")

app = web.Application()
app.router.add_post("/v1/chat/completions", chat)
app.router.add_get("/v1/models", models)
app.router.add_get("/health", health)
web.run_app(app, port=args.port, print=None, access_log=None)


In [ ]:
%%writefile gateway.py
"""Logging API gateway: authenticates by API key, forwards to the model server, streams the answer back,
and logs one line per request: key, arrival time, first byte, stream end, status, bytes.
This is what a provider sees: arrival -> end of stream at its own edge (queueing included)."""
import asyncio, time, argparse
import aiohttp
from aiohttp import web

ap = argparse.ArgumentParser()
ap.add_argument("--port", type=int, default=9000)
ap.add_argument("--backend", default="http://127.0.0.1:8000")
ap.add_argument("--log", default="gateway_log.csv")
args = ap.parse_args()
logf = open(args.log, "a", buffering=1)
if logf.tell() == 0:
    logf.write("key,t_arrival,t_first,t_end,status,nbytes\n")
session = None

async def on_start(app):
    global session
    session = aiohttp.ClientSession(connector=aiohttp.TCPConnector(limit=0), timeout=aiohttp.ClientTimeout(total=None))

async def on_stop(app):
    await session.close(); logf.close()

async def proxy(request):
    key = request.headers.get("Authorization", "").replace("Bearer ", "").strip() or "anon"
    t_arr = time.time(); t_first = None; nbytes = 0; status = 0
    body = await request.read()
    try:
        async with session.post(args.backend + request.path, data=body, headers={"Content-Type": "application/json"}) as r:
            status = r.status
            resp = web.StreamResponse(status=r.status, headers={"Content-Type": r.headers.get("Content-Type", "text/event-stream")})
            await resp.prepare(request)
            async for chunk in r.content.iter_any():
                if t_first is None:
                    t_first = time.time()
                await resp.write(chunk); nbytes += len(chunk)
            await resp.write_eof()
    except (ConnectionResetError, aiohttp.ClientError, asyncio.CancelledError, RuntimeError):
        status = 499                                   # client went away or backend failed mid-stream
        resp = web.Response(status=499)
    finally:
        logf.write(f"{key},{t_arr:.6f},{(t_first or 0):.6f},{time.time():.6f},{status},{nbytes}\n")
    return resp

async def passthrough_get(request):
    async with session.get(args.backend + request.path) as r:
        return web.Response(status=r.status, body=await r.read(), content_type="application/json")

app = web.Application(client_max_size=10 * 1024 ** 2)
app.on_startup.append(on_start); app.on_cleanup.append(on_stop)
app.router.add_post("/v1/chat/completions", proxy)
app.router.add_get("/v1/models", passthrough_get)
web.run_app(app, port=args.port, print=None, access_log=None)


In [ ]:
%%writefile driver.py
"""Workload driver: runs every client population at once against the gateway, each on its own API keys.
Ground truth (key -> workload) is written to labels.json. Client code mirrors how such tools are really written:
the OpenAI SDK with key rotation, LiteLLM's Router over many keys, agent loops, an open-loop relay."""
import asyncio, json, sys, time, itertools, random, math, logging
import numpy as np
from openai import AsyncOpenAI

cfg = json.load(open(sys.argv[1]))
rng = np.random.default_rng(cfg["seed"]); random.seed(cfg["seed"])
BASE = cfg["gateway"] + "/v1"; MODEL = cfg["model"]
T0 = time.time(); T_END = T0 + cfg["run_s"]
labels = {}
stats = {"ok": 0, "err": 0}
err_types = {}
_clients = {}

def keys(prefix, n):
    ks = [f"{prefix}-{i:03d}" for i in range(n)]
    for k in ks: labels[k] = prefix
    return ks

def client(key):
    if key not in _clients:
        _clients[key] = AsyncOpenAI(api_key=key, base_url=BASE, max_retries=0, timeout=300)
    return _clients[key]

def ntok(med=None):
    med = med or cfg["tok_med"]
    return int(np.clip(rng.lognormal(math.log(med), cfg["tok_sig"]), 16, 512))

def prompt():
    return [{"role": "user", "content": f"Write a short note about topic {random.randint(0, 10**6)}."}]

async def call(key, n):
    try:
        s = await client(key).chat.completions.create(model=MODEL, messages=prompt(), max_tokens=n, stream=True,
                                                      extra_body={"ignore_eos": True})
        async for _ in s:
            pass
        stats["ok"] += 1
    except Exception as ex:
        stats["err"] += 1
        k = f"{labels.get(key, '?')}:{type(ex).__name__}:{str(ex)[:80]}"
        err_types[k] = err_types.get(k, 0) + 1
        await asyncio.sleep(0.5)

def alive():
    return time.time() < T_END

# ---------------------------------------------------------------- benign populations
async def human(key):
    h = cfg["human"]
    await asyncio.sleep(rng.exponential(h["gap_s"]))
    while alive():
        for _ in range(rng.geometric(1 / h["mean_req"])):
            if not alive(): return
            await call(key, ntok())
            await asyncio.sleep(rng.lognormal(math.log(h["think_med_s"]), h["think_sig"]))
        await asyncio.sleep(rng.exponential(h["gap_s"]))

async def batch_client(key):          # legitimate single-key closed-loop job
    await asyncio.sleep(rng.uniform(0, 5))
    while alive():
        await call(key, ntok()); await asyncio.sleep(rng.lognormal(math.log(0.2), 0.5))

# ---------------------------------------------------------------- campaign tools
async def rotating_script(ks, K, jitter=0.0):
    """Typical distillation script: K async workers sharing a round-robin key cycle."""
    cyc = itertools.cycle(ks)
    async def worker():
        await asyncio.sleep(rng.uniform(0, 2))
        while alive():
            await call(next(cyc), ntok())
            await asyncio.sleep(rng.lognormal(math.log(0.2), 0.5) + (rng.uniform(0, jitter) if jitter else 0.0))
    await asyncio.gather(*[worker() for _ in range(K)])

async def litellm_router(ks, K, strategy):
    import litellm
    from litellm import Router
    litellm.suppress_debug_info = True
    if hasattr(litellm, "REPEATED_STREAMING_CHUNK_LIMIT"):
        litellm.REPEATED_STREAMING_CHUNK_LIMIT = 10 ** 9   # forced-length generations can repeat; do not abort them
    logging.getLogger("LiteLLM").setLevel(logging.ERROR); logging.getLogger("LiteLLM Router").setLevel(logging.ERROR)
    router = Router(model_list=[{"model_name": "teacher", "litellm_params": {"model": f"openai/{MODEL}", "api_base": BASE, "api_key": k}}
                                for k in ks], routing_strategy=strategy, num_retries=0)
    async def worker():
        await asyncio.sleep(rng.uniform(0, 2))
        while alive():
            try:
                r = await router.acompletion(model="teacher", messages=prompt(), max_tokens=ntok(), stream=True,
                                             extra_body={"ignore_eos": True})
                async for _ in r:
                    pass
                stats["ok"] += 1
            except Exception as ex:
                stats["err"] += 1
                k = f"A3llm:{type(ex).__name__}:{str(ex)[:80]}"
                err_types[k] = err_types.get(k, 0) + 1
                await asyncio.sleep(0.5)
            await asyncio.sleep(rng.lognormal(math.log(0.2), 0.5))
    await asyncio.gather(*[worker() for _ in range(K)])

async def agent_fleet(ks, n_agents):
    """Agents: each episode holds one key; LLM call -> tool execution -> next call."""
    a = cfg["agent"]
    async def agent():
        await asyncio.sleep(rng.uniform(0, 5))
        while alive():
            key = ks[rng.integers(len(ks))]
            for _ in range(rng.geometric(1 / a["mean_steps"])):
                if not alive(): return
                await call(key, ntok(a["tok_med"]))
                await asyncio.sleep(rng.lognormal(math.log(a["tool_med_s"]), 0.6))
            await asyncio.sleep(rng.lognormal(math.log(1.0), 0.5))
    await asyncio.gather(*[agent() for _ in range(n_agents)])

async def open_loop_relay(ks, lam):
    """Relay forwarding independent end-user requests: arrivals do not wait for completions."""
    tasks = set()
    while alive():
        await asyncio.sleep(rng.exponential(1 / lam))
        t = asyncio.create_task(call(ks[rng.integers(len(ks))], ntok()))
        tasks.add(t); t.add_done_callback(tasks.discard)
    if tasks:
        await asyncio.wait(tasks, timeout=120)

async def main():
    w = cfg["workloads"]; jobs = []
    jobs += [human(k) for k in keys("hum", w["humans"])]
    jobs += [batch_client(k) for k in keys("bat", w["batch_clients"])]
    jobs.append(rotating_script(keys("A1sem", 40), w["A1_K"]))
    jobs.append(rotating_script(keys("A2jit", 40), w["A2_K"], jitter=w["A2_jitter_s"]))
    jobs.append(litellm_router(keys("A3llm", 40), w["A3_K"], "simple-shuffle"))
    jobs.append(agent_fleet(keys("A4agt", 40), w["A4_agents"]))
    jobs.append(open_loop_relay(keys("A5rel", 40), w["A5_rate"]))
    jobs.append(rotating_script(keys("Ccomp", 10), w["C_K"]))     # legitimate company: one queue over 10 keys
    json.dump(labels, open("labels.json", "w"))

    async def progress():
        while alive():
            await asyncio.sleep(60)
            print(f"[driver] t={time.time() - T0:.0f}s ok={stats['ok']} err={stats['err']}", flush=True)
    await asyncio.gather(*jobs, progress())
    print(f"[driver] done ok={stats['ok']} err={stats['err']}", flush=True)
    top = sorted(err_types.items(), key=lambda kv: -kv[1])[:8]
    print("[driver] errors " + json.dumps(dict(top)), flush=True)
    json.dump(err_types, open("driver_errors.json", "w"))

asyncio.run(main())


In [ ]:
# Cell 4 — client libraries
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "aiohttp", "openai", "litellm"], check=False)
import importlib; importlib.invalidate_caches()
import aiohttp, openai
print("aiohttp", aiohttp.__version__, "openai", openai.__version__)


In [ ]:
# Cell 5 — model server: vLLM on the GPU if possible, otherwise the simulated server
import urllib.request
def wait_http(url, timeout_s, proc=None):
    t = time.time()
    while time.time() - t < timeout_s:
        if proc is not None and proc.poll() is not None:
            return False
        try:
            urllib.request.urlopen(url, timeout=5); return True
        except Exception:
            time.sleep(5)
    return False

def tail(path, n=25):
    try:
        return "".join(open(path, errors="ignore").readlines()[-n:])
    except Exception:
        return ""

BACKEND = {"kind": None, "attempts": []}
def gpu_present():
    try:
        return subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    except Exception:
        return False

def vllm_import_check(py):
    r = subprocess.run([py, "-c", "import torch, vllm; print('vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available())"],
                       capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr).strip()[-1200:]

def launch_vllm(py, extra_env, extra_args):
    env = dict(os.environ, **extra_env)
    log = open("vllm.log", "w")
    p = subprocess.Popen([py, "-m", "vllm.entrypoints.openai.api_server", "--model", CFG["model"], "--dtype", "half",
                          "--max-model-len", "2048", "--gpu-memory-utilization", "0.85", "--max-num-seqs", "128", "--port", "8000"] + extra_args,
                         stdout=log, stderr=subprocess.STDOUT, env=env)
    ok = wait_http("http://127.0.0.1:8000/v1/models", 900, p)
    if not ok:
        p.kill(); time.sleep(5)
    return ok, p

if CFG["backend"] in ("auto", "vllm") and gpu_present():
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
    pythons = []
    # Route 1: Kaggle's own Python. vLLM upgrades torch; Kaggle's torchaudio then no longer matches, so remove it (not needed for text models).
    t_inst = time.time()
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], capture_output=True, text=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"], capture_output=True, text=True)
    ok, msg = vllm_import_check(sys.executable)
    BACKEND["route1"] = dict(install_ok=r.returncode == 0, import_ok=ok, minutes=round((time.time() - t_inst) / 60, 1), msg=msg[-300:])
    print("route 1 (system python):", "OK" if ok else "FAILED", "\n", msg)
    if ok:
        pythons.append(sys.executable)
    else:
        # Route 2: an isolated virtual environment, so nothing preinstalled can clash with vLLM's own torch
        t_inst = time.time()
        venv = "/tmp/vllm_env"
        subprocess.run([sys.executable, "-m", "venv", venv], check=False)
        py = os.path.join(venv, "bin", "python")
        subprocess.run([py, "-m", "pip", "install", "-q", "--upgrade", "pip"], capture_output=True)
        r2 = subprocess.run([py, "-m", "pip", "install", "-q", "vllm"], capture_output=True, text=True)
        ok2, msg2 = vllm_import_check(py)
        BACKEND["route2"] = dict(install_ok=r2.returncode == 0, import_ok=ok2, minutes=round((time.time() - t_inst) / 60, 1), msg=msg2[-300:])
        print("route 2 (isolated venv):", "OK" if ok2 else "FAILED", "\n", msg2, "\n", r2.stderr[-800:])
        if ok2:
            pythons.append(py)
    for py in pythons:
        for extra_env, extra_args in (({}, []), ({}, ["--enforce-eager"]), ({"VLLM_ATTENTION_BACKEND": "TRITON_ATTN"}, ["--enforce-eager"])):
            ok, p = launch_vllm(py, extra_env, extra_args)
            BACKEND["attempts"].append(dict(python=py, env=extra_env, args=extra_args, ok=ok, log_tail=None if ok else tail("vllm.log", 15)[-1500:]))
            if ok:
                BACKEND.update(kind="vllm", proc=p, model=CFG["model"]); break
            print("vLLM launch failed:", extra_env, extra_args, "\n", tail("vllm.log", 15))
        if BACKEND["kind"]:
            break
if BACKEND["kind"] is None:
    if CFG["backend"] == "vllm":
        raise RuntimeError("vLLM requested but could not start; see BACKEND['attempts']")
    p = subprocess.Popen([sys.executable, "simserver.py", "--port", "8000"], stdout=open("sim.log", "w"), stderr=subprocess.STDOUT)
    assert wait_http("http://127.0.0.1:8000/health", 60, p), tail("sim.log")
    BACKEND.update(kind="sim", proc=p, model="sim")
print("backend:", BACKEND["kind"], [(a["env"], a["args"], a["ok"]) for a in BACKEND["attempts"]])


In [ ]:
# Cell 6 — gateway + smoke test through the real OpenAI SDK
for f in ("gateway_log.csv", "labels.json"):
    if os.path.exists(f): os.remove(f)
gw = subprocess.Popen([sys.executable, "gateway.py", "--port", "9000", "--backend", "http://127.0.0.1:8000", "--log", "gateway_log.csv"],
                      stdout=open("gateway.out", "w"), stderr=subprocess.STDOUT)
assert wait_http(CFG["gateway"] + "/v1/models", 60, gw), tail("gateway.out")
from openai import OpenAI
c = OpenAI(api_key="smoke-000", base_url=CFG["gateway"] + "/v1")
t = time.time()
s = c.chat.completions.create(model=BACKEND["model"], messages=[{"role": "user", "content": "Say hello."}], max_tokens=96, stream=True,
                              extra_body={"ignore_eos": True})
n = sum(1 for _ in s)
BACKEND["smoke"] = dict(chunks=n, seconds=round(time.time() - t, 2))
print("smoke test:", BACKEND["smoke"])


In [ ]:
# Cell 7 — run every population at once
dcfg = dict(CFG, model=BACKEND["model"])
json.dump(dcfg, open("driver_cfg.json", "w"))
t_run = time.time()
drv = subprocess.Popen([sys.executable, "driver.py", "driver_cfg.json"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in drv.stdout:
    if line.startswith("[driver]"): print(line.rstrip(), flush=True)
drv.wait()
RUN = dict(driver_rc=drv.returncode, wall_min=round((time.time() - t_run) / 60, 1))
print(RUN)


In [ ]:
# Cell 8 — analysis with ground truth
import numpy as np, pandas as pd
from occlib import *
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(CFG["seed"])
log = pd.read_csv("gateway_log.csv")
labels = json.load(open("labels.json"))
log["label"] = log.key.map(labels)
ok = (log.status == 200) & log.label.notna()
RES = {"cfg": CFG, "backend": {k: v for k, v in BACKEND.items() if k != "proc"}, "run": RUN,
       "log": dict(n=len(log), ok_frac=round(float(ok.mean()), 4), by_label=log[ok].label.value_counts().to_dict())}
d = log[ok].copy()
t0 = d.t_arrival.min() + CFG["warmup_s"]; t1 = d.t_end.max() - 30
d = d[(d.t_end > t0) & (d.t_arrival < t1)]
d["dur"] = d.t_end - d.t_arrival
RES["durations_s"] = d.groupby("label").dur.median().round(2).to_dict()
keys_by = d.groupby("label").key.unique().to_dict()
kid = {k: i for i, k in enumerate(sorted(d.key.unique()))}
d["acct"] = d.key.map(kid)

def pool(keys_):
    m = d.key.isin(keys_)
    acc, _ = relabel(d.acct.values[m])
    return acc, d.t_arrival.values[m], d.t_end.values[m]

TAUS = (20.0, 60.0, 180.0)
def stats(keys_, n_sur=0):
    acc, s, e = pool(keys_)
    o, fk, b = occupancy(acc, s, e, t0, t1, None), fano_khat(acc, s, e, t0, t1), baselines(acc, s, e, t0, t1)
    out = dict(n_keys=len(keys_), req_h=round(b["req_per_h"]), peak=b["peak_N"], fano=round(float(fk["F"]), 3),
               R=round(float(o["R"]), 3), K_hat=round(float(fk["K_hat"]), 2) if np.isfinite(fk["K_hat"]) else None)
    for tau in TAUS:
        out[f"R{int(tau)}"] = round(float(occupancy(acc, s, e, t0, t1, tau)["R"]), 3) if (t1 - t0) >= 2 * tau else float("nan")
    if n_sur:
        sur = surrogate_null(acc, s, e, t0, t1, CFG["tau"], n_sur, rng)
        r = out[f"R{int(CFG['tau'])}"]
        out["p_low"] = round(float((1 + (sur <= r).sum()) / (n_sur + 1)), 4)     # turn-taking (R below independence)
        out["p_high"] = round(float((1 + (sur >= r).sum()) / (n_sur + 1)), 4)    # bunching (R above independence)
    return out

hum = list(keys_by.get("hum", [])); bat = list(keys_by.get("bat", []))
null = [stats(list(rng.choice(hum, min(40, len(hum)), replace=False))) for _ in range(200)]
NULL = {k: np.array([x[k] for x in null]) for k in ["R"] + [f"R{int(t)}" for t in TAUS]}
RES["null_humans_40"] = {k: dict(mean=round(float(v.mean()), 4), sd=round(float(v.std()), 4), q01=round(float(np.quantile(v, 0.01)), 4),
                                 q99=round(float(np.quantile(v, 0.99)), 4)) for k, v in NULL.items()}
def add_z(st):
    for k, v in NULL.items():
        st[f"z_{k}"] = round(float((v.mean() - st[k]) / v.std()), 1)      # positive = turn-taking, negative = bunching
    return st
pools = {}
for lab in ["A1sem", "A2jit", "A3llm", "A4agt", "A5rel", "Ccomp"]:
    if lab in keys_by:
        pools[lab] = add_z(stats(list(keys_by[lab]), n_sur=300))
heavy = [stats(list(rng.choice(hum, min(36, len(hum)), replace=False)) + bat) for _ in range(30)]
pools["benign_36hum_4batch"] = {k: round(float(np.mean([h[k] for h in heavy])), 3) for k in ["R", "R20", "R60", "fano", "req_h"]}
pools["benign_36hum_4batch"]["R20_min"] = round(float(np.min([h["R20"] for h in heavy])), 3)
for lab in ["A1sem", "A3llm", "A4agt"]:
    if lab in keys_by:
        camp = list(keys_by[lab])
        mixes = [add_z(stats(list(rng.choice(camp, min(20, len(camp)), replace=False)) + list(rng.choice(hum, min(20, len(hum)), replace=False)))) for _ in range(10)]
        pools[f"mix_random20{lab}_20hum"] = {k: round(float(np.median([m[k] for m in mixes])), 3)
                                             for k in ["R", "R20", "R60", "R180", "z_R", "z_R20", "z_R60", "z_R180"]}
# round-robin 'travelling wave': contiguous vs random halves of the rotated key ring
if "A1sem" in keys_by:
    ring = sorted(keys_by["A1sem"])
    pools["A1sem_ring_contiguous_half"] = add_z(stats(ring[:20]))
    pools["A1sem_ring_random_half"] = add_z(stats(list(rng.choice(ring, min(20, len(ring)), replace=False))))
RES["pools"] = pools
# theory check: R predicted from the pool's own Fano factor, p = 1 - F
th = {}
for lab in ["A1sem", "A2jit", "A3llm"]:
    if lab in pools:
        p_ = 1 - pools[lab]["fano"]; M = pools[lab]["n_keys"]
        th[lab] = dict(R_observed=pools[lab]["R"], R_predicted=round((1 - p_) / (1 - p_ / M), 3), K_hat=pools[lab]["K_hat"])
RES["theory_check"] = th
# pool growth: seed = half the campaign + 20 humans; rank the rest of the campaign among the other humans
growth = {}
for lab in ["A1sem", "A2jit", "A3llm", "A4agt"]:
    if lab not in keys_by: continue
    camp = list(keys_by[lab]); hh = list(rng.permutation(hum))
    seed_keys = camp[:20] + hh[:20]; cand_keys = camp[20:] + hh[20:]
    ms = d.key.isin(seed_keys).values; mc = d.key.isin(cand_keys).values
    cidx = {k: i for i, k in enumerate(cand_keys)}
    sc = corr_with_pool((d.acct.values[ms], d.t_arrival.values[ms], d.t_end.values[ms]),
                        (d.key[mc].map(cidx).values, d.t_arrival.values[mc], d.t_end.values[mc]), t0, t1, tau=CFG["tau"], n_cand=len(cand_keys))
    y = np.r_[np.ones(len(camp) - 20), np.zeros(len(hh) - 20)]
    growth[lab] = round(float(roc_auc_score(y, np.nan_to_num(-sc, nan=0.0))), 3)
RES["pool_growth_auc"] = growth
# serving-side effects: does latency rise with load (which would couple honest users)?
allv = d.sort_values("t_arrival")
tt = np.concatenate([allv.t_arrival.values, allv.t_end.values]); dd = np.concatenate([np.ones(len(allv)), -np.ones(len(allv))])
oo = np.lexsort((dd, tt)); lvl = np.cumsum(dd[oo])
level_at = pd.Series(lvl, index=tt[oo]).groupby(level=0).last()
n_at = level_at.reindex(allv.t_arrival.values, method="ffill").values
per_byte = allv.dur.values / np.maximum(allv.nbytes.values, 1)
RES["serving"] = dict(site_inflight_median=float(np.median(n_at)), site_inflight_p95=float(np.quantile(n_at, 0.95)),
                      spearman_latency_per_byte_vs_inflight=round(float(spearmanr(n_at, per_byte)[0]), 3))
RES["go_no_go"] = "GO" if all(pools.get(l, {}).get(f"R{int(CFG['tau'])}", 1) < 0.9 for l in ["A1sem", "A3llm"]) else "STOP"
print(json.dumps(RES, indent=1, default=str))


In [ ]:
# Cell 9 — shut down and print the summary
for proc in (gw, BACKEND.get("proc")):
    try: proc.terminate()
    except Exception: pass
d.to_csv("nb5_requests.csv", index=False)
RES["runtime_min"] = round((time.time() - T_START) / 60, 1)
json.dump(RES, open("nb5_results.json", "w"), indent=1, default=str)
print("===== PASTE FROM HERE =====")
print(json.dumps(RES, separators=(",", ":"), default=str))
print("===== PASTE TO HERE =====")
